# DataQuest 2026 — Limpeza dos dados WPP2024
Este notebook limpa o arquivo **Medium** da ONU e gera duas tabelas:
- `paises_limpo` → só países/áreas (237), 1950–2100, com Região/Sub-região e colunas do suplemento
- `agregados_limpo` → regiões, grupos de renda, mundo etc. (**nunca somar com os países**)

**Como usar:** `Ambiente de execução > Executar tudo`. Na primeira célula, escolha de onde vêm os arquivos.

## 1. Onde estão os arquivos?
Coloque os 3 arquivos abaixo numa pasta do Google Drive (ou envie direto no Colab):
- `WPP2024_Demographic_Indicators_Medium_csv.gz`
- `WPP2024_Supplement_Births_and_LE60.csv`

In [ ]:
import os, pandas as pd

# >>> ESCOLHA: "drive" (pasta do Google Drive) ou "upload" (enviar agora) <<<
MODO = "drive"
PASTA_DRIVE = "/content/drive/MyDrive/DataQuest/"   # ajuste para a pasta de vocês

try:
    import google.colab  # só existe no Colab
    if MODO == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        PASTA = SAIDA = PASTA_DRIVE
    else:
        from google.colab import files
        files.upload()          # selecione os 2 arquivos
        PASTA = SAIDA = "/content/"
except ImportError:
    PASTA = os.environ.get("WPP_PASTA", "./")   # fora do Colab
    SAIDA = os.environ.get("WPP_SAIDA", "./")

MEDIUM = PASTA + "WPP2024_Demographic_Indicators_Medium_csv.gz"
SUPL   = PASTA + "WPP2024_Supplement_Births_and_LE60.csv"
print("Arquivos encontrados:", os.path.exists(MEDIUM), os.path.exists(SUPL))

## 2. Carregar
`keep_default_na=False` evita que a **Namíbia** (ISO2 = `"NA"`) seja lida como valor ausente.

In [ ]:
df = pd.read_csv(MEDIUM, compression="gzip", encoding="utf-8-sig", low_memory=False,
                 keep_default_na=False, na_values=[""])
for c in ["Notes", "ISO3_code", "ISO2_code", "LocTypeName"]:
    df[c] = df[c].replace("", pd.NA)
print(df.shape)
df.head(3)

## 3. Limpeza
- O ano **2101** está vazio em todos os indicadores → removido
- Cerca de 234 grupos "ad hoc" vêm sem tipo → rotulados
- `Location` **não é único** (Australia/New Zealand e Latin America and the Caribbean aparecem 2×) → use sempre `LocID` como chave
- Nova coluna `Periodo`: Estimativa (≤2023) ou Projecao (≥2024)

In [ ]:
df = df[df["Time"] <= 2100].copy()
df["LocTypeName"] = df["LocTypeName"].fillna("Ad Hoc groups")
df["Periodo"] = df["Time"].apply(lambda t: "Estimativa" if t <= 2023 else "Projecao")
df.LocTypeName.value_counts()

## 4. Separar países de agregados e criar Região/Sub-região
Usa o `ParentID` da hierarquia da ONU. Para a América do Norte (EUA, Canadá, Bermuda, Groenlândia, St. Pierre e Miquelon) o código-pai 918 não existe como linha no arquivo, então é preenchido manualmente.

In [ ]:
loc = df.drop_duplicates("LocID").set_index("LocID")
sub = loc[loc.LocTypeName == "Subregion"]["Location"]
reg = loc[loc.LocTypeName == "Geographic region"]["Location"]
sub_parent = loc[loc.LocTypeName == "Subregion"]["ParentID"]

paises = df[df.LocTypeName == "Country/Area"].copy()
paises["Subregiao"] = paises["ParentID"].map(sub)
paises["Regiao"] = paises["ParentID"].map(sub_parent).map(reg)
na = paises["ParentID"] == 918
paises.loc[na, ["Subregiao", "Regiao"]] = "Northern America"

## 5. Juntar o suplemento
Adiciona `BirthsMale`, `BirthsFemale` e `LE60*`. O Vaticano não existe no suplemento (ficará vazio).

In [ ]:
sup = pd.read_csv(SUPL, encoding="utf-8-sig", keep_default_na=False, na_values=[""])
sup = sup.drop(columns=["ISO3_code", "Location"])
paises = paises.merge(sup, on=["LocID", "Time"], how="left")

drop = ["SortOrder", "SDMX_code", "LocTypeID", "ParentID", "VarID", "Variant", "Notes"]
paises = paises.drop(columns=drop)
agregados = df[df.LocTypeName != "Country/Area"].drop(columns=drop + ["ISO3_code", "ISO2_code"])
print("paises:", paises.shape, "| agregados:", agregados.shape)

## 6. Validação
A soma dos países deve ser igual ao total "World" em todos os anos.

In [ ]:
mundo = agregados[agregados.LocTypeName == "World"].set_index("Time").TPopulation1July
soma = paises.groupby("Time").TPopulation1July.sum()
print("Maior diferença países vs. mundo (%):", float(((soma/mundo - 1).abs()*100).max()))
print("Países:", paises.LocID.nunique(), "| Namíbia ISO2:", paises.loc[paises.ISO3_code=="NAM","ISO2_code"].iloc[0])
print()
print(paises.drop_duplicates("LocID").Regiao.value_counts())
print()
nulos = paises.isna().sum(); print(nulos[nulos > 0])

**Sobre os valores nulos (não são erros):**
- `DoublingTime`: vazio quando a população não cresce
- `BirthsMale/Female`, `LE60*`: só faltam no Vaticano

**Outliers** como expectativa de vida de ~11 anos são eventos históricos reais (guerras, genocídios) — confira país e ano antes de excluir.

## 7. Salvar (opcional)

In [ ]:
paises.to_csv(SAIDA + "paises_limpo.csv", index=False)
agregados.to_csv(SAIDA + "agregados_limpo.csv", index=False)
print("Salvo em", SAIDA)

## 8. Pronto para analisar
`paises` e `agregados` já estão na memória. Exemplo: população mundial por região.

In [ ]:
import matplotlib.pyplot as plt
p = paises.pivot_table(index="Time", columns="Regiao", values="TPopulation1July", aggfunc="sum") / 1e6
ax = p.plot(figsize=(10,5), title="População por região (milhões)")
ax.axvline(2023, color="gray", ls="--"); ax.text(2025, p.values.max()*0.95, "projeção →")
plt.show()